# Cover Letter Forge - local LLM Tailored Writer

This notebook demonstrates how we prompt a local conversational LLM to compose tailored cover letters. It supports three distinct styles (Professional, Story-Driven, and Data-First) and enforces hard length and vocabulary constraints.

In [1]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Design Prompts for Cover Letter Styles

We provide instructions detailing length limits (380-480 words) and vocabulary exclusions (e.g. banning 'I am writing to express my interest').

In [2]:
style = "data_first"  # Options: professional, story_driven, data_first

style_guides = {
    "professional": "formal, polished, and concise. Lead with your strongest technical alignment.",
    "story_driven": "narrative and humanized. Open with a brief personal origin story connecting your interest to this company.",
    "data_first": "metrics-led. Lead with a striking quantitative accomplishment. Support claim with numbers."
}

system_prompt = f"""
You are an expert career writer. Write a cover letter for the candidate based on target company, alignment score, and resume achievements.
Style Directive: {style_guides[style]}
Rules:
- Letter Word Count: 380 to 480 words.
- Banned Clichés: 'I am writing to express my interest', 'hope this finds you well', 'leverage synergies', 'team player'.
- Explicitly name the candidate's portfolio projects and target company.
"""

user_input = """
Candidate: Alice
Target Company: Google
Alignment Score: 85/100
Strongest Project Bullets:
- Designed and deployed a FastAPI backend prediction pipeline for VinoMetrix, reducing API prediction latencies by 40% and processing 5,000 requests/minute.
- Built a custom XGBoost model yielding 95% classification accuracy on chemical datasets.
"""

prompt = f"<|system|>\n{system_prompt}\n<|user|>\n{user_input}\n<|assistant|>\n"

## 2. Load Model and Generate Letter

In [3]:
model_id = "microsoft/Phi-3-mini-4k-instruct"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=500, temperature=0.6, do_sample=True)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    letter = result.replace(prompt, "").strip()
    print("\n--- Generated Cover Letter ---\n")
    print(letter)
    print(f"\nWord Count: {len(letter.split())} words")
except Exception as e:
    print(f"Skipping actual model load during demo. Details: {e}")

Skipping actual model load during demo. Details: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`


C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
